# SWOT water levels at a site
Search, select granules, download and extract. Each row is one file. Tiles can overlap on a single pass.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from SWOT_toolbox.catalog import search_site, download_selected, extract_selection

site = "Belt Bay"
lon, lat = 137.028098, -28.893022
start, end = "2026-01-01", "2026-01-31"  # inclusive UTC dates
resolution = 100
directory = Path("data/rasters")  # folder containing the NetCDF files
output = Path("output") / site / f"{start}_{end}_{resolution}m"
output.mkdir(parents=True, exist_ok=True)

## Search the catalogue

In [ ]:
inventory = search_site(lon, lat, start, end, directory, resolution)
inventory.to_csv(output / "catalog.csv", index=False)
display(inventory[["date_utc", "cycle", "pass", "tile", "local_state", "size_bytes"]])

## Select and download
Keep all rows or set indices to a list, e.g. [0, 2]. Set download to True to retrieve missing or invalid files using Earthdata Login.

In [ ]:
indices = inventory.index
selected = inventory.loc[indices].copy()
fetch = selected.local_state.ne("available")
print(f"{fetch.sum()} files to retrieve ({selected.loc[fetch, 'size_bytes'].sum()/1e6:.1f} MB)")
download = False
if download:
    selected = download_selected(selected)
    display(selected[["filename", "local_state", "download_error"]])

## Extract and audit
The example averages quality classes 0 and 1 in a 1300 m window without temporal outlier rejection. Adjust the settings for your site. Extend the dates and rerun the cells to update the series.

In [ ]:
audit = extract_selection(selected, lon, lat, directory, width_m=1300,
                          levels=(0, 1), quality_mode="pixel", iqr="off")
audit.to_csv(output / "site_audit.csv", index=False)
display(audit[["date_utc", "pass", "tile", "local_state", "n_valid", "status", "wse"]])

In [ ]:
usable = audit.loc[audit.status.eq("accepted")]
if usable.empty:
    print("No accepted water levels. Check local_state and status.")
else:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.scatter(pd.to_datetime(usable.date_utc, utc=True), usable.wse, s=22, color="#236581")
    ax.set(xlabel="Date (UTC)", ylabel="Water surface elevation (m)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(output / "water_levels.pdf")